# Mexico City landslide social-risk replication

This notebook reproduces the **published calculation workflow** in Novelo-Casanova et al. (2022), *The Risk Atlas of Mexico City, Mexico*. It produces a landslide-susceptibility raster and combines a five-class hazard input with social vulnerability to create the paper's qualitative social-risk score. It is a transparent methodological replication, not a claim to reproduce the authors' exact map without their source layers and ArcGIS settings.

Paper workflow: `geology, slope, relative height, land use/vegetation -> weighted susceptibility (1-5)`. Separately, `hazard class (1-5) × social-vulnerability class (1-5) -> social-risk score (1-25) -> five risk classes`. For landslides, the article documents susceptibility but does not publish a separate temporal-probability hazard model; using susceptibility class as the hazard input is therefore an explicit proxy assumption, not a result reproduced from the paper.

The paper uses a 5 m DEM; a 1972-2018 landslide/rockfall inventory; geology, slope, relative-height, land-use and vegetation inputs; and 13 social-vulnerability indicators weighted using the Analytic Hierarchy Process.

## Required inputs

Place the following in `data/raw/`, all in a projected metre-based CRS and covering the same area. The notebook stops on unmapped categories rather than silently assigning them a risk score.

- `dem_5m.tif`: DEM in metres (the paper used 5 m resolution).
- `geology.gpkg`: polygons with categorical `unit`.
- `land_cover.gpkg`: polygons with categorical `class`.
- `social_vulnerability.tif`: raster whose cells are classes 1-5. Create it from the 13 indicators in the optional section below, or supply an independently prepared, paper-compatible classified raster.
- `landslide_hazard.tif` (optional but required for a strict risk calculation): a hazard raster classified 1-5. If omitted, the notebook uses the calculated susceptibility class as a clearly labelled hazard proxy.
- `landslides.gpkg` (optional): points or polygons for an independent presence-versus-background diagnostic.

Install if necessary: `%pip install geopandas rasterio scipy matplotlib scikit-learn`

In [ ]:
from pathlib import Path
import unicodedata
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import rasterio
from rasterio.features import rasterize
from rasterio.plot import plotting_extent
from rasterio.warp import reproject, Resampling
from scipy.ndimage import minimum_filter

ROOT = Path.cwd()
RAW, OUT = ROOT / 'data' / 'raw', ROOT / 'data' / 'processed'
OUT.mkdir(parents=True, exist_ok=True)
DEM_PATH = RAW / 'dem_5m.tif'
GEOLOGY_PATH = RAW / 'geology.gpkg'
LAND_COVER_PATH = RAW / 'land_cover.gpkg'
SOCIAL_VULNERABILITY_PATH = RAW / 'social_vulnerability.tif'
LANDSLIDES_PATH = RAW / 'landslides.gpkg'
LANDSLIDE_HAZARD_PATH = RAW / 'landslide_hazard.tif'  # optional: strict risk input
for path in (DEM_PATH, GEOLOGY_PATH, LAND_COVER_PATH, SOCIAL_VULNERABILITY_PATH):
    if not path.exists(): raise FileNotFoundError(f'Missing required input: {path}')

In [ ]:
# Tables 3 and 4 of the paper. Scores: 1 = very low susceptibility; 5 = very high.
WEIGHTS = {'geology': .60, 'slope': .20, 'relative_height': .10, 'land_cover': .10}
assert np.isclose(sum(WEIGHTS.values()), 1)
GEOLOGY_SCORES = {
 'lake alluvial deposits':1, 'basalt (quaternary)':2, 'andesite (quaternary)':2, 'basaltic andesite (quaternary)':2,
 'alluvial deposits (foothill)':3, 'andesite':3, 'basalt':3, 'andesitic-basaltic (altered lavas)':3, 'dacite':3, 'basaltic andesite (cones)':3,
 'andesite (very altered and fractured rocks)':4, 'basalt (very altered and fractured rocks)':4, 'basaltic andesite (very altered and fractured rocks)':4, 'dacite (very altered and fractured rocks)':4, 'basaltic andesite (very altered cones)':4, 'basalt (very altered cones)':4, 'lahar deposit':4, 'pumice flow':4, 'avalanche':4,
 'andesite (very altered rocks with fractures and faults)':5, 'dacite (very altered rocks with fractures and faults)':5, 'lahar deposit (very altered rocks with fractures and faults)':5, 'pumice flow and volcanic ash':5, 'alluvial deposits (slope)':5}
LAND_COVER_SCORES = {'body of water':1, 'hydrophilic and halophilic':1, 'tule':1, 'oyamel forest':2, 'sarcocaulous shrubland':3, 'cultivated grassland':3, 'halophile grassland':3, 'induced grassland':3, 'secondary shrub vegetation of oyamel forest':3, 'no apparent vegetation':4, 'annual irrigation agriculture':4, 'urban zone':4, 'human settlements':5}
def norm(value): return ' '.join(unicodedata.normalize('NFKD', str(value)).encode('ascii','ignore').decode().lower().split())
def slope_class(x):
    # Table 4 omits 30-45 degrees. It is assigned class 4, a stated assumption.
    return np.select([x <= 5, x <= 15, x <= 30, x <= 60, x > 60], [1,2,3,4,5], default=np.nan).astype('float32')
def height_class(x): return np.select([x <= 10, x <= 20, x <= 50, x <= 100, x > 100], [1,2,3,4,5], default=np.nan).astype('float32')

In [ ]:
# Use the DEM as the common analysis grid and derive slope and relative height.
with rasterio.open(DEM_PATH) as src:
    dem = src.read(1, masked=True).astype('float32'); profile = src.profile.copy()
    transform, crs = src.transform, src.crs
if not crs or not crs.is_projected: raise ValueError('DEM must have a projected CRS in metres.')
z = dem.filled(np.nan); cell_x, cell_y = abs(transform.a), abs(transform.e)
dy, dx = np.gradient(z, cell_y, cell_x)
slope_score = slope_class(np.degrees(np.arctan(np.hypot(dx, dy))))
# The paper does not define the relative-height neighbourhood. This editable 500 m radius is an assumption.
RELIEF_RADIUS_M = 500
window = max(3, int(round(2 * RELIEF_RADIUS_M / cell_x)) + 1); window += (window % 2 == 0)
local_min = minimum_filter(np.where(np.isnan(z), np.inf, z), size=window, mode='nearest')
relative_height_score = height_class(z - local_min)

geology, cover = gpd.read_file(GEOLOGY_PATH).to_crs(crs), gpd.read_file(LAND_COVER_PATH).to_crs(crs)
def raster_scores(frame, field, lookup):
    if field not in frame: raise KeyError(f'{field!r} not found: {frame.columns.tolist()}')
    scores = frame[field].map(norm).map(lookup)
    unknown = sorted(frame.loc[scores.isna(), field].dropna().astype(str).unique())
    if unknown: raise ValueError(f'Unmapped {field} categories: {unknown}')
    return rasterize(zip(frame.geometry, scores.astype('float32')), out_shape=dem.shape, transform=transform, fill=np.nan, dtype='float32')
geology_score = raster_scores(geology, 'unit', GEOLOGY_SCORES)
cover_score = raster_scores(cover, 'class', LAND_COVER_SCORES)

In [ ]:
# Landslide susceptibility (Table 3), then classify its 1-5 weighted score into five equal-width intervals.
susceptibility = (.60 * geology_score + .20 * slope_score + .10 * relative_height_score + .10 * cover_score).astype('float32')
valid = np.isfinite(z) & np.isfinite(susceptibility)
susceptibility[~valid] = np.nan
hazard_class = np.select([susceptibility < 1.8, susceptibility < 2.6, susceptibility < 3.4, susceptibility < 4.2, susceptibility <= 5], [1,2,3,4,5], default=0).astype('uint8')
# The paper states five susceptibility categories but not its precise GIS class breaks; equal-width breaks are explicit and replaceable.
profile.update(count=1, dtype='float32', nodata=-9999, compress='deflate')
with rasterio.open(OUT / 'landslide_susceptibility_score.tif', 'w', **profile) as dst: dst.write(np.where(valid, susceptibility, -9999), 1)
profile.update(dtype='uint8', nodata=0)
with rasterio.open(OUT / 'landslide_hazard_class.tif', 'w', **profile) as dst: dst.write(hazard_class, 1)

In [ ]:
# Paper section 11: multiply a five-class hazard raster by a five-class social-vulnerability raster.
# The landslide section publishes susceptibility, not a distinct temporal-probability hazard surface.
# Therefore we use an external hazard raster when supplied; otherwise susceptibility is an explicit proxy.
hazard_input_path = LANDSLIDE_HAZARD_PATH if LANDSLIDE_HAZARD_PATH.exists() else None
if hazard_input_path is None:
    risk_hazard_class = hazard_class
    print('WARNING: No landslide_hazard.tif supplied. Using susceptibility class as a hazard proxy.')
else:
    with rasterio.open(hazard_input_path) as src:
        hazard_resampled = np.zeros(dem.shape, dtype='float32')
        reproject(source=rasterio.band(src, 1), destination=hazard_resampled, src_transform=src.transform, src_crs=src.crs, dst_transform=transform, dst_crs=crs, resampling=Resampling.nearest, dst_nodata=np.nan)
    if not np.all(np.isin(hazard_resampled[np.isfinite(hazard_resampled)], [1,2,3,4,5])): raise ValueError('landslide_hazard.tif must contain only class values 1-5.')
    risk_hazard_class = np.where(np.isfinite(hazard_resampled), hazard_resampled, 0).astype('uint8')
with rasterio.open(SOCIAL_VULNERABILITY_PATH) as src:
    social = np.zeros(dem.shape, dtype='float32')
    reproject(source=rasterio.band(src, 1), destination=social, src_transform=src.transform, src_crs=src.crs, dst_transform=transform, dst_crs=crs, resampling=Resampling.nearest, dst_nodata=np.nan)
if not np.all(np.isin(social[np.isfinite(social)], [1,2,3,4,5])): raise ValueError('social_vulnerability.tif must contain only class values 1-5.')
risk_score = risk_hazard_class.astype('float32') * social
risk_score[(risk_hazard_class == 0) | ~np.isfinite(social)] = np.nan
# Table 6: 1-5 very low; 6-10 low; 11-15 moderate; 16-20 high; 21-25 very high.
risk_class = np.select([risk_score <= 5, risk_score <= 10, risk_score <= 15, risk_score <= 20, risk_score <= 25], [1,2,3,4,5], default=0).astype('uint8')
profile.update(dtype='float32', nodata=-9999)
with rasterio.open(OUT / 'landslide_social_risk_score.tif', 'w', **profile) as dst: dst.write(np.where(np.isfinite(risk_score), risk_score, -9999), 1)
profile.update(dtype='uint8', nodata=0)
with rasterio.open(OUT / 'landslide_social_risk_class.tif', 'w', **profile) as dst: dst.write(risk_class, 1)
{name: int((risk_class == code).sum()) for code, name in enumerate(['', 'very low', 'low', 'moderate', 'high', 'very high']) if code}

In [ ]:
# Visual QA: inspect the final class surface before using it.
fig, ax = plt.subplots(figsize=(10, 9))
im = ax.imshow(np.ma.masked_equal(risk_class, 0), extent=plotting_extent(risk_class, transform), cmap='YlOrRd', vmin=1, vmax=5)
fig.colorbar(im, ax=ax, ticks=[1,2,3,4,5], label='Social-risk class (1 = very low, 5 = very high)')
ax.set(title='Mexico City landslide social risk', xlabel='Easting (m)', ylabel='Northing (m)'); plt.show()

## Optional: deriving social vulnerability from source indicators

The paper uses `SV = sum(SVi * wi)` for 13 AGEB-level indicators and then Jenks Natural Breaks to assign five social-vulnerability classes. Its published weights are: `[0.069, 0.080, 0.065, 0.110, 0.107, 0.117, 0.058, 0.087, 0.040, 0.107, 0.055, 0.074, 0.030]` for SV1-SV13 respectively. The paper's table appears to print SV2 as `0.80`; `0.080` is used here because the published weights otherwise do not sum to 1. Derive directionally consistent, normalized indicator rasters; calculate the weighted sum; classify it with Jenks; and save classes 1-5 as `social_vulnerability.tif`. Document indicator definitions, Census vintage, normalization direction, Jenks implementation, and resulting breaks.

## Interpretation and limits

- This is a qualitative likelihood-of-social-risk index, not a calibrated probability or expected-loss model.
- A strict risk calculation requires an independently derived landslide-hazard class. If `landslide_hazard.tif` is absent, the notebook calculates *susceptibility* and uses it only as a transparent hazard proxy; this is not the same as calculating hazard.
- The paper describes kriging interpolation after the hazard-vulnerability overlay. This notebook preserves the aligned raster overlay directly; add interpolation only with a defensible point-support and variogram workflow.
- The article does not specify the 30-45 degree slope score, relative-height neighbourhood, original susceptibility breaks, or original GIS layers. Those implementation choices must be sensitivity-tested.

Reference: Novelo-Casanova, D. A. et al. (2022). *The Risk Atlas of Mexico City, Mexico: a tool for decision-making and disaster prevention*. Natural Hazards, 111, 411-437. https://doi.org/10.1007/s11069-021-05059-z